# Base de Modelagem Diária

Este notebook cria a primeira base oficial para previsão diária de faturamento bruto por filial.

Decisões desta versão:

- alvo: `faturamento_bruto_dia`;
- granularidade: `codigo_filial + data`;
- split temporal fixo;
- remoção de colunas com vazamento de informação;
- manutenção de calendário, cadastro e histórico defasado (`lags` e médias móveis com `shift(1)`).


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display


def encontrar_raiz_projeto(inicio=None):
    inicio = Path.cwd() if inicio is None else Path(inicio)
    candidatos = [inicio, *inicio.parents]

    for candidato in candidatos:
        if (candidato / "Base_V2").exists():
            return candidato

    raise FileNotFoundError(
        "Nao foi possivel encontrar a raiz do projeto. "
        "Execute o notebook a partir da raiz ou mantenha a pasta Base_V2."
    )


ROOT = encontrar_raiz_projeto()
BASE_V2 = ROOT / "Base_V2"
BASE_MODELAGEM = ROOT / "Base_Modelagem"
BASE_MODELAGEM.mkdir(parents=True, exist_ok=True)

ARQUIVO_FEATURES = BASE_V2 / "features_filiais_diarias_V2.parquet"
ARQUIVO_BASE_MODELAGEM = BASE_MODELAGEM / "base_modelagem_diaria.parquet"
ARQUIVO_FEATURES_JSON = BASE_MODELAGEM / "features_modelagem.json"

ALVO = "faturamento_bruto_dia"

DATA_CORTE_TREINO = pd.Timestamp("2025-08-31")
DATA_CORTE_VALIDACAO = pd.Timestamp("2025-10-31")

COLUNAS_CATEGORICAS = [
    "codigo_filial",
    "dia_semana",
    "faixa_vida",
    "localidade",
    "uf",
    "tipo_estabelecimento",
    "delivery",
    "panvel_clinic",
    "estacionamento",
    "atendimento_24_horas",
    "grupo_metragem",
    "idade_filial",
]

COLUNAS_NUMERICAS = [
    "ano",
    "mes",
    "dia_mes",
    "semana_mes",
    "semana_ano",
    "dia_semana_id",
    "trimestre",
    "semestre",
    "eh_semana_5",
    "fim_semana_id",
    "eh_feriado_bancario",
    "eh_dia_util",
    "ordem_dia_util_mes",
    "dias_uteis_no_mes",
    "eh_primeiro_dia_util_mes",
    "eh_quinto_dia_util_mes",
    "eh_inicio_mes",
    "eh_fim_mes",
    "dias_no_mes",
    "periodo_ordem",
    "dias_desde_primeira_venda",
    "metragem_area_venda",
    "idade_filial_meses",
    "faturamento_bruto_lag_1d",
    "faturamento_bruto_lag_7d",
    "faturamento_bruto_lag_14d",
    "faturamento_bruto_lag_28d",
    "cupons_lag_7d",
    "quantidade_lag_7d",
    "faturamento_med_lag_7d",
    "faturamento_n_med_lag_7d",
    "share_med_faturamento_lag_7d",
    "faturamento_bruto_media_movel_7d",
    "faturamento_bruto_std_movel_7d",
    "faturamento_bruto_media_movel_28d",
    "faturamento_bruto_std_movel_28d",
    "cupons_media_movel_28d",
    "quantidade_media_movel_28d",
    "faturamento_med_media_movel_28d",
    "faturamento_n_med_media_movel_28d",
]

COLUNAS_VAZAMENTO_EXCLUIDAS = [
    "cupons_dia",
    "quantidade_dia",
    "ticket_medio_bruto_dia",
    "itens_por_cupom_dia",
    "faturamento_med_dia",
    "faturamento_n_med_dia",
    "quantidade_med_dia",
    "quantidade_n_med_dia",
    "teve_venda",
    "share_med_faturamento",
    "share_n_med_faturamento",
    "share_med_quantidade",
    "share_n_med_quantidade",
    "faturamento_bruto_ratio_mes_dia_semana",
    "faturamento_bruto_media_mes_semana_dia",
    "faturamento_bruto_std_mes_semana_dia",
    "faturamento_bruto_obs_mes_semana_dia",
    "faturamento_bruto_cv_mes_semana_dia",
    "faturamento_bruto_ratio_mes_semana_dia",
]

COLUNAS_PERFIL_COMPLETO_EXCLUIDAS = [
    "dias_total",
    "dias_com_venda_total",
    "faturamento_bruto_total",
    "faturamento_bruto_medio_dia",
    "faturamento_bruto_mediano_dia",
    "faturamento_bruto_std_dia",
    "faturamento_bruto_max_dia",
    "faturamento_bruto_min_dia",
    "quantidade_total",
    "quantidade_media_dia",
    "cupons_total",
    "cupons_medios_dia",
    "ticket_medio_bruto_dia_medio",
    "itens_por_cupom_dia_medio",
    "faturamento_med_total",
    "faturamento_n_med_total",
    "quantidade_med_total",
    "quantidade_n_med_total",
    "share_med_faturamento_medio",
    "share_n_med_faturamento_medio",
    "share_med_quantidade_medio",
    "share_n_med_quantidade_medio",
    "faturamento_bruto_media_calendario_medio",
    "faturamento_bruto_std_calendario_medio",
    "faturamento_bruto_cv_calendario_medio",
    "faturamento_bruto_ratio_calendario_medio",
    "faturamento_bruto_obs_calendario_medio",
    "faturamento_bruto_obs_calendario_min",
    "faturamento_bruto_obs_calendario_max",
    "slots_calendario_total",
    "pct_dias_com_venda",
    "cv_faturamento_bruto_dia",
    "share_med_faturamento_total",
    "share_n_med_faturamento_total",
    "share_med_quantidade_total",
    "share_n_med_quantidade_total",
]


## Ler e Selecionar Colunas

As features permitidas são conhecidas antes do dia previsto ou vêm de histórico defasado. Colunas do próprio dia e perfis calculados usando a série completa ficam fora desta primeira base para evitar vazamento.


In [2]:
features = pd.read_parquet(ARQUIVO_FEATURES)
features["data"] = pd.to_datetime(features["data"])
features["codigo_filial"] = features["codigo_filial"].astype(str)

colunas_categoricas = [coluna for coluna in COLUNAS_CATEGORICAS if coluna in features.columns]
colunas_numericas = [coluna for coluna in COLUNAS_NUMERICAS if coluna in features.columns]
colunas_features = colunas_categoricas + colunas_numericas

faltantes = sorted(set(COLUNAS_CATEGORICAS + COLUNAS_NUMERICAS) - set(colunas_features))
if faltantes:
    print("Colunas planejadas ausentes:", faltantes)

colunas_features_sem_chave = [coluna for coluna in colunas_features if coluna != "codigo_filial"]

base_modelagem = features[["codigo_filial", "data", ALVO] + colunas_features_sem_chave].copy()
base_modelagem["target_log1p"] = np.log1p(base_modelagem[ALVO].clip(lower=0))

for coluna in colunas_categoricas:
    base_modelagem[coluna] = base_modelagem[coluna].astype("object").fillna("SEM_INFO").astype(str)

for coluna in colunas_numericas:
    base_modelagem[coluna] = pd.to_numeric(base_modelagem[coluna], errors="coerce").astype("float64")

base_modelagem[colunas_numericas] = base_modelagem[colunas_numericas].replace([np.inf, -np.inf], np.nan).fillna(0)

base_modelagem["conjunto"] = np.select(
    [
        base_modelagem["data"].le(DATA_CORTE_TREINO),
        base_modelagem["data"].le(DATA_CORTE_VALIDACAO),
    ],
    ["treino", "validacao"],
    default="teste",
)

ordem_colunas = [
    "codigo_filial",
    "data",
    "conjunto",
    ALVO,
    "target_log1p",
] + [coluna for coluna in colunas_features if coluna not in ["codigo_filial"]]

base_modelagem = base_modelagem[ordem_colunas].sort_values(["codigo_filial", "data"]).reset_index(drop=True)

resumo = (
    base_modelagem.groupby("conjunto", as_index=False)
    .agg(
        linhas=("data", "size"),
        filiais=("codigo_filial", "nunique"),
        data_min=("data", "min"),
        data_max=("data", "max"),
        faturamento_total=(ALVO, "sum"),
        faturamento_medio=(ALVO, "mean"),
    )
)

display(resumo)
display(pd.DataFrame({
    "grupo": ["categoricas", "numericas", "total_features", "colunas_vazamento_excluidas", "colunas_perfil_completo_excluidas"],
    "qtd": [
        len(colunas_categoricas),
        len(colunas_numericas),
        len(colunas_features),
        len([c for c in COLUNAS_VAZAMENTO_EXCLUIDAS if c in features.columns]),
        len([c for c in COLUNAS_PERFIL_COMPLETO_EXCLUIDAS if c in features.columns]),
    ],
}))
display(base_modelagem.head())


,conjunto,linhas,filiais,data_min,data_max,faturamento_total,faturamento_medio
0,teste,6100,100,2025-11-01,2025-12-31,5.522794e+08,90537.608990
1,treino,60882,100,2024-01-01,2025-08-31,4.554780e+09,74813.237107
2,validacao,6100,100,2025-09-01,2025-10-31,5.132620e+08,84141.305041


,grupo,qtd
0,categoricas,12
1,numericas,40
2,total_features,52
3,colunas_vazamento_excluidas,19
4,colunas_perfil_completo_excluidas,36


,codigo_filial,data,conjunto,faturamento_bruto_dia,target_log1p,dia_semana,faixa_vida,localidade,uf,tipo_estabelecimento,...,faturamento_n_med_lag_7d,share_med_faturamento_lag_7d,faturamento_bruto_media_movel_7d,faturamento_bruto_std_movel_7d,faturamento_bruto_media_movel_28d,faturamento_bruto_std_movel_28d,cupons_media_movel_28d,quantidade_media_movel_28d,faturamento_med_media_movel_28d,faturamento_n_med_media_movel_28d
0,1500,2024-01-02,treino,48449.24,10.788293,terca-feira,MAIS DE 3 ANOS,CURITIBA,PR,CENTRO,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0.0000
1,1500,2024-01-03,treino,53257.89,10.882920,quarta-feira,MAIS DE 3 ANOS,CURITIBA,PR,CENTRO,...,0.0,0.0,48449.240000,0.000000,48449.240000,0.000000,240.000000,1617.00,31029.200000,17420.0400
2,1500,2024-01-04,treino,57909.36,10.966652,quinta-feira,MAIS DE 3 ANOS,CURITIBA,PR,CENTRO,...,0.0,0.0,50853.565000,3400.229023,50853.565000,3400.229023,246.500000,1650.00,32859.665000,17993.9000
3,1500,2024-01-05,treino,66140.99,11.099559,sexta-feira,MAIS DE 3 ANOS,CURITIBA,PR,CENTRO,...,0.0,0.0,53205.496667,4730.277624,53205.496667,4730.277624,241.666667,1692.00,34539.206667,18666.2900
4,1500,2024-01-06,treino,31027.62,10.342665,sabado,MAIS DE 3 ANOS,CURITIBA,PR,CENTRO,...,0.0,0.0,56439.370000,7533.177586,56439.370000,7533.177586,252.750000,1812.75,36078.117500,20361.2525


## Salvar Base

Salva a base e um JSON com a lista de features para os notebooks seguintes.


In [3]:
metadata_features = {
    "alvo": ALVO,
    "target_transformado": "target_log1p",
    "data_corte_treino": str(DATA_CORTE_TREINO.date()),
    "data_corte_validacao": str(DATA_CORTE_VALIDACAO.date()),
    "features_categoricas": colunas_categoricas,
    "features_numericas": colunas_numericas,
    "features_modelo": colunas_features,
    "colunas_vazamento_excluidas": [c for c in COLUNAS_VAZAMENTO_EXCLUIDAS if c in features.columns],
    "colunas_perfil_completo_excluidas": [c for c in COLUNAS_PERFIL_COMPLETO_EXCLUIDAS if c in features.columns],
}

base_modelagem.to_parquet(ARQUIVO_BASE_MODELAGEM, index=False)
ARQUIVO_FEATURES_JSON.write_text(json.dumps(metadata_features, indent=2, ensure_ascii=False), encoding="utf-8")

display(pd.Series({
    "base_modelagem": str(ARQUIVO_BASE_MODELAGEM),
    "features_json": str(ARQUIVO_FEATURES_JSON),
}).to_frame("arquivo"))


,arquivo
base_modelagem,E:\trabalhos\PROJ3- Panvel\Base_Modelagem\base...
features_json,E:\trabalhos\PROJ3- Panvel\Base_Modelagem\feat...
